# Difference Mapping Social Vulnerability across Census Units in Colorado
SEES 3050 Final Project · Elif Korkmaz

This notebook runs the full workflow. The functions are in `sovi_maup.py` and all settings are in `config.toml`.

**Before running:** put your Census API key in a file named `census_api_key.txt` in this folder (one line, no quotes), or set the `CENSUS_API_KEY` environment variable. Downloads are saved in `data/raw` and reused on later runs.

## Setup

In [1]:
import pandas as pd
import sovi_maup as sm

cfg = sm.load_config("config.toml")   # all settings
sm.start_log(cfg)                      # messages go to the screen and logs/run.log
api_key = sm.get_api_key()

19:47:59  INFO  ---- New run ----


## Step 1. Prepare data
Downloads the ACS tables and TIGER/Line boundaries for each scale, computes the indicators and their margins of error, and removes units with no population or missing values. The first run takes a few minutes because block groups are downloaded county by county.

In [2]:
labels = sm.download_variable_labels(cfg)   # official ACS labels of every variable used
labels.to_csv(f"{cfg['paths']['tables']}/variable_labels.csv", index=False)
labels.head(10)

,code,label,concept
0,B01001_001,Estimate!!Total:,Sex by Age
1,B01001_003,Estimate!!Total:!!Male:!!Under 5 years,Sex by Age
2,B01001_004,Estimate!!Total:!!Male:!!5 to 9 years,Sex by Age
3,B01001_005,Estimate!!Total:!!Male:!!10 to 14 years,Sex by Age
4,B01001_006,Estimate!!Total:!!Male:!!15 to 17 years,Sex by Age
5,B01001_020,Estimate!!Total:!!Male:!!65 and 66 years,Sex by Age
6,B01001_021,Estimate!!Total:!!Male:!!67 to 69 years,Sex by Age
7,B01001_022,Estimate!!Total:!!Male:!!70 to 74 years,Sex by Age
8,B01001_023,Estimate!!Total:!!Male:!!75 to 79 years,Sex by Age
9,B01001_024,Estimate!!Total:!!Male:!!80 to 84 years,Sex by Age


In [3]:
data, avail, keep, screening = sm.prepare_data(cfg, api_key)

19:47:59  INFO  == Preparing scale: block group ==
19:47:59  INFO  Using saved file data/raw/acs2024_CO_bg.csv
19:48:00  INFO  block group: 4058 polygons, projected to EPSG:5070
19:48:00  INFO  == Preparing scale: tract ==
19:48:00  INFO  Using saved file data/raw/acs2024_CO_tract.csv
19:48:00  INFO  tract: 1447 polygons, projected to EPSG:5070
19:48:00  INFO  == Preparing scale: county ==
19:48:00  INFO  Using saved file data/raw/acs2024_CO_county.csv
19:48:00  INFO  county: 64 polygons, projected to EPSG:5070
19:48:00  INFO  Indicators kept: 24 of 25. Dropped: ['QNRRES']
19:48:00  INFO  block group: {'units': 4058, 'dropped_no_population': 21, 'dropped_missing_values': 27, 'kept_units': 4010}
19:48:00  INFO  tract: {'units': 1447, 'dropped_no_population': 10, 'dropped_missing_values': 9, 'kept_units': 1428}
19:48:00  INFO  county: {'units': 64, 'dropped_no_population': 0, 'dropped_missing_values': 0, 'kept_units': 64}


### Indicator availability
Share of populated units with a missing value. An indicator is kept only if it is at most 20% missing at every scale.

In [4]:
avail.to_csv(f"{cfg['paths']['tables']}/indicator_availability.csv")
avail.round(3)

,block group,tract,county,kept
MEDAGE,0.001,0.000,0.0,True
QAGEDEP,0.000,0.000,0.0,True
QFEMALE,0.000,0.000,0.0,True
QASIAN,0.000,0.000,0.0,True
QBLACK,0.000,0.000,0.0,True
QNATAM,0.000,0.000,0.0,True
QHISP,0.000,0.000,0.0,True
QESL,0.004,0.004,0.0,True
QFHH,0.004,0.004,0.0,True
QSPHH,0.006,0.006,0.0,True


### Units removed during screening

In [5]:
screening.to_csv(f"{cfg['paths']['tables']}/screening_report.csv")
screening

,units,dropped_no_population,dropped_missing_values,kept_units
block group,4058,21,27,4010
tract,1447,10,9,1428
county,64,0,0,64


### Reliability of the kept indicators
Median coefficient of variation (CV, %) and the share of units with CV above 40% (low reliability).

In [6]:
reliability = sm.reliability_table(data, keep, cfg)
reliability.to_csv(f"{cfg['paths']['tables']}/indicator_reliability.csv")
reliability.round(2)

median_cv               share_cv_over_40             
scale     block group county  tract      block group county tract
indicator                                                        
MEDAGE          10.79   1.28   5.77             0.02   0.00  0.00
PERCAP          14.13   5.03   8.86             0.02   0.00  0.00
PPUNIT          24.59   2.23  10.99             0.08   0.00  0.00
QAGEDEP         15.06   4.28   8.73             0.07   0.00  0.00
QASIAN          74.14  27.19  53.23             0.58   0.31  0.66
QBLACK          78.67  35.11  63.50             0.46   0.39  0.61
QCOSTBUR        30.40  10.01  18.23             0.23   0.00  0.01
QCVLUN          69.15  26.56  48.12             0.74   0.22  0.71
QED12LES        62.55  16.78  42.73             0.66   0.09  0.53
QESL            97.64  41.39  79.29             0.36   0.47  0.60
QEXTRCT         94.60  17.97  75.09             0.37   0.14  0.65
QFEMALE         11.10   1.12   6.10             0.03   0.00  0.00
QFHH            58.63  19.86  36.79             0.73   0.06  0.38
QHISP           42.30   0.00  24.14             0.53   0.06  0.14
QMOHO           60.99  15.85  50.87             0.18   0.03  0.27
QNATAM          95.00  44.39  89.89             0.22   0.48  0.44
QNOAUTO         78.53  26.85  56.09             0.57   0.19  0.70
QPOVTY          54.92  15.26  36.09             0.73   0.00  0.38
QRENTER         34.13   9.25  18.79             0.39   0.00  0.11
QRICH200K       42.51  17.66  27.69             0.47   0.14  0.26
QSERV           41.56  12.93  25.54             0.52   0.00  0.08
QSPHH           67.98  22.14  43.63             0.70   0.19  0.58
QSSBEN          27.89   4.66  15.22             0.23   0.00  0.03
QVACANT         71.44   9.85  53.07             0.52   0.00  0.66

## O0. Describe the indicators
Statistics of each kept indicator at each scale, one map figure per indicator (state and Denver metro, same color range on every panel), and the correlation between indicators at the block-group level.

In [7]:
stats = sm.describe_indicators(data, keep)
stats.to_csv(f"{cfg['paths']['tables']}/indicator_statistics.csv")
stats.round(2).head(12)

,,n,mean,sd,min,median,max
indicator,scale,,,,,,
MEDAGE,block group,4010,40.60,9.80,15.00,39.40,83.30
QAGEDEP,block group,4010,36.91,12.00,0.00,37.58,92.23
QFEMALE,block group,4010,49.57,6.75,11.27,49.61,79.42
QASIAN,block group,4010,2.90,4.88,0.00,1.05,56.53
QBLACK,block group,4010,3.40,7.20,0.00,0.13,77.22
QNATAM,block group,4010,0.38,2.02,0.00,0.00,74.49
QHISP,block group,4010,22.02,19.45,0.00,16.05,100.00
QESL,block group,4010,2.47,5.47,0.00,0.00,66.94
QFHH,block group,4010,9.05,8.65,0.00,6.93,83.14


In [8]:
sm.plot_indicator_maps(data, keep, cfg)    # saved in outputs/maps/indicators/
corr = sm.plot_correlation(data["block group"]["values"], cfg, "block group")

/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/plotting.py:1408: UserWarning: The GeoSeries you are attempting to plot is empty. Nothing has been displayed.
  return plot_dataframe(data, *args, **kwargs)
/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/plotting.py:1408: UserWarning: The GeoSeries you are attempting to plot is empty. Nothing has been displayed.
  return plot_dataframe(data, *args, **kwargs)
/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/plotting.py:1408: UserWarning: The GeoSeries you are attempting to plot is empty. Nothing has been displayed.
  return plot_dataframe(data, *args, **kwargs)
/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/plotting.py:1408: UserWarning: The GeoSeries you are attempting to plot is empty. Nothing has been displayed.
  return plot_dataframe(data, *args, **kwargs)
/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/pl

## Step 2. Build the index (O1)
For each scale: z-scores, PCA on the correlation matrix, components with eigenvalue > 1, varimax rotation, component directions set by the expected direction of the indicators with loadings of 0.5 or more, equal-weight sum, and percentile rank. The additive index (sum of direction-adjusted z-scores) and the fixed-structure index (block-group components applied to every scale) are built in the same loop as checks.

In [9]:
structures = sm.build_all_indices(data, cfg)
summary = sm.structure_table(structures)
summary.to_csv(f"{cfg['paths']['tables']}/index_structure.csv")
for scale, s in structures.items():
    s["loadings"].round(3).to_csv(f"{cfg['paths']['tables']}/loadings_{sm.SCALE_CODE[scale]}.csv")
summary

19:48:29  INFO  block group: 6 components kept, 56.9% of variance explained
19:48:29  INFO  tract: 6 components kept, 64.8% of variance explained
19:48:29  INFO  county: 7 components kept, 77.6% of variance explained


,components,variance_explained,main_indicators
scale,,,
block group,6,56.9,"PC1(-): QRICH200K, QASIAN, PERCAP | PC2(-): QS..."
tract,6,64.8,"PC1(-): PERCAP, QRICH200K, QSERV, QHISP | PC2(..."
county,7,77.6,"PC1(+): QRICH200K, QMOHO, QEXTRCT, PERCAP | PC..."


In [10]:
# Agreement between the PCA index and the two checks (Spearman rank correlation)
checks = pd.DataFrame({
    scale: {"PCA vs additive": d["index"]["score"].corr(d["additive"]["score"], method="spearman"),
            "PCA vs fixed structure": d["index"]["score"].corr(d["fixed"]["score"], method="spearman")}
    for scale, d in data.items()})
checks.to_csv(f"{cfg['paths']['tables']}/index_checks.csv")
checks.round(3)

,block group,tract,county
PCA vs additive,0.888,0.785,0.810
PCA vs fixed structure,1.000,0.858,0.747


In [11]:
sm.plot_index_maps(data, cfg)                                   # PCA-based index
sm.plot_index_maps(data, cfg, "additive", "Additive index")     # robustness check

/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/plotting.py:1408: UserWarning: The GeoSeries you are attempting to plot is empty. Nothing has been displayed.
  return plot_dataframe(data, *args, **kwargs)
/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/plotting.py:1408: UserWarning: The GeoSeries you are attempting to plot is empty. Nothing has been displayed.
  return plot_dataframe(data, *args, **kwargs)
/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/plotting.py:1408: UserWarning: The GeoSeries you are attempting to plot is empty. Nothing has been displayed.
  return plot_dataframe(data, *args, **kwargs)
/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/plotting.py:1408: UserWarning: The GeoSeries you are attempting to plot is empty. Nothing has been displayed.
  return plot_dataframe(data, *args, **kwargs)


PosixPath('outputs/maps/index_maps_additive.png')

## Step 3. Map the differences (O2)
**Monte Carlo.** Each indicator is drawn 500 times from Normal(value, MOE/1.645), and the index is rebuilt each time with the same components and directions as the main run. This gives 500 percentile ranks per unit.

In [12]:
sm.run_simulations(data, structures, cfg)   # about 15 seconds

19:48:39  INFO  block group: 500 simulations in 2 s; median width of 90% rank interval = 30.8 points
19:48:40  INFO  tract: 500 simulations in 1 s; median width of 90% rank interval = 33.0 points
19:48:40  INFO  county: 500 simulations in 0 s; median width of 90% rank interval = 17.2 points


**Difference maps.** For each pair of scales, every smaller unit gets the rank of the larger unit that contains it (read from the GEOID), and the difference is small rank minus large rank. Positive (red) = the smaller unit is more vulnerable than the larger unit's map shows. A difference is *reliable* when its 90% simulation interval does not include 0. White = no population or missing data; gray (right panels) = difference within sampling error.

In [13]:
tables = {}
for small, large in sm.scale_pairs(cfg):
    table = sm.difference_table(data, small, large)
    table = sm.add_reliability(table, data, small, large)
    table.to_csv(f"{cfg['paths']['processed']}/differences_{sm.SCALE_CODE[small]}_{sm.SCALE_CODE[large]}.csv")
    sm.plot_difference_maps(data, table, small, large, cfg)
    tables[(small, large)] = table

diff_summary = pd.DataFrame({f"{s} - {l}": sm.summarize_differences(t) for (s, l), t in tables.items()})
diff_summary.to_csv(f"{cfg['paths']['tables']}/difference_summary.csv")
diff_summary.round(1)

/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/plotting.py:1408: UserWarning: The GeoSeries you are attempting to plot is empty. Nothing has been displayed.
  return plot_dataframe(data, *args, **kwargs)
/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/plotting.py:1408: UserWarning: The GeoSeries you are attempting to plot is empty. Nothing has been displayed.
  return plot_dataframe(data, *args, **kwargs)
/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/plotting.py:1408: UserWarning: The GeoSeries you are attempting to plot is empty. Nothing has been displayed.
  return plot_dataframe(data, *args, **kwargs)
/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/plotting.py:1408: UserWarning: The GeoSeries you are attempting to plot is empty. Nothing has been displayed.
  return plot_dataframe(data, *args, **kwargs)
/opt/anaconda3/envs/sees3050-final/lib/python3.11/site-packages/geopandas/pl

,block group - tract,block group - county,tract - county
units,4010.0,4010.0,1428.0
min,-87.0,-77.9,-78.7
max,83.1,92.1,78.8
mean,-1.1,-4.8,-4.5
sd,23.4,30.9,29.4
mean_abs,17.5,25.4,24.1
area_within_5,14.0,16.9,25.8
area_within_10,26.6,30.4,36.1
area_over_20,53.3,54.4,44.5
area_higher_than_large,86.7,68.5,50.3


**Distributions.** Share of the state's land area and population in each 5-point bin of rank difference.

In [14]:
sm.plot_difference_histograms(tables, cfg)

PosixPath('outputs/maps/difference_histograms.png')

## Step 4. Compare scales (O3)
For each pair of scales: Spearman correlation between a unit's rank and its larger unit's rank; agreement on the most vulnerable 20% (by units and by people); size and reliability of the differences; and the robustness checks, which repeat the difference map with the additive index and with the fixed-structure index (`diff_corr_*` = rank correlation between that difference map and the main one).

In [15]:
comparison = sm.compare_scales(data, tables, cfg)
comparison.to_csv(f"{cfg['paths']['tables']}/scale_comparison.csv")
comparison.round(2).T

pair,block group - tract,block group - county,tract - county
spearman_rank,0.67,0.25,0.32
top20_agree_units,59.10,24.94,22.73
top20_agree_pop,59.37,21.17,18.10
mean_abs_diff,17.47,25.43,24.11
reliable_units,26.66,54.54,50.28
reliable_pop,25.21,57.30,52.36
mean_abs_diff_additive,12.93,23.78,22.46
diff_corr_additive,0.66,0.83,0.76
mean_abs_diff_fixed,13.55,26.82,25.76
diff_corr_fixed,0.74,0.83,0.81


## Tests
**Nesting test.** A point inside every smaller polygon must fall inside the polygon of the larger unit given by its GEOID. This checks the join used for the difference maps against the geometry.

In [16]:
nesting = sm.check_nesting(data, cfg)
nesting.to_csv(f"{cfg['paths']['tables']}/test_nesting.csv")
nesting

,small_units,parent_found_pct,point_inside_parent_pct
pair,,,
block group - tract,4058,100.0,100.0
block group - county,4058,100.0,100.0
tract - county,1447,100.0,100.0


**Hand check.** For two random units per scale, the table lists the raw ACS estimates used by four indicators, the formula, and the value computed by the code. A few of these values were looked up on data.census.gov (ACS 5-year 2024, same table and GEOID) and recomputed with a calculator.

In [17]:
hand = sm.hand_check_table(data, cfg)
hand

,scale,GEOID,indicator,formula,acs_values,code_value,code_moe
0,block group,080050074002,QPOVTY,(C17002_002 + C17002_003) / C17002_001 * 100,C17002_002 = 23 ± 28; C17002_003 = 112 ± 188; ...,16.52,22.17
1,block group,080050074002,QRENTER,B25003_003 / B25003_001 * 100,B25003_003 = 81 ± 51; B25003_001 = 282 ± 106,28.72,14.51
2,block group,080050074002,PERCAP,B19301_001,"B19301_001 = 31,408 ± 7,159",31408.00,7159.00
3,block group,080050074002,QNOAUTO,(B25044_003 + B25044_010) / B25044_001 * 100,B25044_003 = 6 ± 10; B25044_010 = 7 ± 10; B250...,4.61,4.71
4,block group,080050800002,QPOVTY,(C17002_002 + C17002_003) / C17002_001 * 100,C17002_002 = 65 ± 93; C17002_003 = 0 ± 14; C17...,4.87,6.92
5,block group,080050800002,QRENTER,B25003_003 / B25003_001 * 100,B25003_003 = 23 ± 28; B25003_001 = 414 ± 81,5.56,6.68
6,block group,080050800002,PERCAP,B19301_001,"B19301_001 = 49,324 ± 13,986",49324.00,13986.00
7,block group,080050800002,QNOAUTO,(B25044_003 + B25044_010) / B25044_001 * 100,B25044_003 = 12 ± 19; B25044_010 = 0 ± 14; B25...,2.90,5.67
8,tract,08041002501,QPOVTY,(C17002_002 + C17002_003) / C17002_001 * 100,C17002_002 = 71 ± 56; C17002_003 = 191 ± 146; ...,7.27,4.21
9,tract,08041002501,QRENTER,B25003_003 / B25003_001 * 100,"B25003_003 = 164 ± 78; B25003_001 = 1,522 ± 221",10.78,4.88
